In [13]:
import json
import argparse
import os


path = '/workspace/term_basev2_1203/KG_RAG_WORK/src/GraphRAG/dataset/all_triples/'
input_path = f"{path}combine_test_triples_0326_v2_1.json"
# 读取 JSON 文件
def read_json_file(input_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# 提取头实体和尾实体，并去重
def extract_entities(input_path):
    data = read_json_file(input_path)
    entities = set()  # 使用集合去重
    for triple_group in data:  # 遍历最外层列表
        for triple in triple_group:  # 遍历每个三元组
            head_entity = triple[0]  # 头实体
            tail_entity = triple[2]  # 尾实体
            entities.add(head_entity)
            entities.add(tail_entity)
    return list(entities)  # 转换为列表返回

docs = extract_entities(input_path)
print("去重后的头实体和尾实体：")
print(docs[0])

1
去重后的头实体和尾实体：
按月


In [25]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("/workspace/bge-large-zh-v1.5")
dense_dim = 1024  # 根据网页1模型参数定义
# 生成向量
docs1 = ["文本1", "文本2"]
# 截断文本（假设最大长度为 512 字符）
max_length = 512
docs = [doc[:max_length] for doc in docs]
docs_embeddings = model.encode(docs,normalize_embeddings=True,batch_size=32)
dense_dim = len(docs_embeddings[0])

In [26]:
len(docs_embeddings)

1185

In [28]:
len(docs)

1185

In [52]:
from pymilvus import (
    connections,
    utility,
    FieldSchema,
    CollectionSchema,
    DataType,
    Collection,
)

connections.connect(uri="./milvus.db")

fields = [
    # Use auto generated id as primary key
    FieldSchema(
        name="pk", dtype=DataType.VARCHAR, is_primary=True, auto_id=True, max_length=100
    ),
    # Store the original text to retrieve based on semantically distance
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=512),
    # Milvus now supports both sparse and dense vectors,
    # we can store each in a separate field to conduct hybrid search on both vectors
    # FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=dense_dim),
]
schema = CollectionSchema(fields)

col_name = "embedding_demo"
if utility.has_collection(col_name):
    Collection(col_name).drop()
col = Collection(col_name, schema, consistency_level="Strong")

# sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
# col.create_index("sparse_vector", sparse_index)
dense_index = {"index_type": "AUTOINDEX", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()


In [44]:
# 4. 删除集合
# col.drop()

In [58]:
!du -sh ./milvus.db

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


20M	./milvus.db


In [54]:

for i in range(0, len(docs), 50):
    batched_entities = [
        docs[i : i + 50],
        # docs_embeddings["sparse"][i : i + 50],
        # docs_embeddings["dense"][i : i + 50],
        docs_embeddings[i : i + 50].tolist()  # 转换为列表格式
    ]
    col.insert(batched_entities)
print("Number of entities inserted:", col.num_entities)


Number of entities inserted: 1185


In [ ]:
question_path = '/workspace/term_basev2_1203/KG_RAG_WORK/src/GraphRAG/dataset/TERM_BASE_QA/test2_question1218.json'
with open(question_path) as f:
        dataset = json.load(f)
for i, data in tqdm(enumerate(dataset), total=len(dataset)):
    question = dataset[i]["question"]
    answer = dataset[i]["answer"]
    
    print('answer:',result)
    # pipeline(i)
    res.append(result)